In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.model_selection import StratifiedKFold
from sktime.transformations.panel.rocket import MiniRocketMultivariate
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data

In [ ]:
seed = 1
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
n_components = 32
n_splits = 5

In [ ]:
rng = np.random.RandomState(seed)

In [ ]:
data = pd.read_parquet("../data/GBG500.parquet")
data

In [ ]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

In [ ]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [ ]:
# sktime expects (n_instances, n_channels, n_timepoints)
data_panel = data_np_clean.transpose(0, 2, 1)

y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
detectors = ['lof', 'iso_forest', 'ocsvm']
scores = {k: np.full(len(y_true), np.nan) for k in detectors}
scores_pca = {k: np.full(len(y_true), np.nan) for k in detectors}

for fold, (train_idx, test_idx) in enumerate(skf.split(data_panel, y_true)):
    minirocket = MiniRocketMultivariate(random_state=rng.randint(1000), n_jobs=1)
    minirocket.fit(data_panel[train_idx])
    Z_train_raw = minirocket.transform(data_panel[train_idx])
    Z_test_raw = minirocket.transform(data_panel[test_idx])

    scaler = StandardScaler()
    Z_train_sc = scaler.fit_transform(Z_train_raw)
    Z_test_sc = scaler.transform(Z_test_raw)

    pca = PCA(n_components=n_components, random_state=rng.randint(1000))
    Z_train_pca = pca.fit_transform(Z_train_sc)
    Z_test_pca = pca.transform(Z_test_sc)

    for Z_train, Z_test, s in [(Z_train_sc, Z_test_sc, scores),
                                (Z_train_pca, Z_test_pca, scores_pca)]:
        lof = LocalOutlierFactor(n_neighbors=20, novelty=True)
        lof.fit(Z_train)
        s['lof'][test_idx] = -lof.score_samples(Z_test)

        iso = IsolationForest(random_state=rng.randint(1000))
        iso.fit(Z_train)
        s['iso_forest'][test_idx] = -iso.score_samples(Z_test)

        ocsvm = OneClassSVM(kernel='rbf')
        ocsvm.fit(Z_train)
        s['ocsvm'][test_idx] = -ocsvm.decision_function(Z_test)

    print(f"Fold {fold+1}/{n_splits} done")


In [ ]:
for label, s, prefix in [('MiniRocket', scores, 'GBG500_ap_minirocket'),
                          ('MiniRocket+PCA', scores_pca, 'GBG500_ap_minirocket_pca')]:
    for key, suffix in [('lof', 'lof'), ('iso_forest', 'iso_forest'), ('ocsvm', 'ocsvm')]:
        ap = average_precision_score(y_true, s[key])
        print(f"{label}+{key} AP = {ap:.4f}")
        sb.glue(f"{prefix}_{suffix}", float(ap))